# ¿Qué tramos de Granada se vuelven críticos con −15 % de velocidad, y son los mismos que en seco?

Investigación con el paquete `trafico_lluvia`.

In [2]:
  import sys
  from pathlib import Path

  RAIZ = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
  sys.path.insert(0, str(RAIZ))

  import pandas as pd
  import trafico_lluvia as tl
  from trafico_lluvia.escenarios import Escenario

  G = tl.cargar_red()
  muestra = tl.cargar_muestra(G)
  tl.verificar_muestra(G, muestra)

  # Seco = None. El −15 % se define como Escenario propio: (central, mín, máx).
  UNIFORME = Escenario("uniforme_15", "-", (0.15,) * 3, (0.15,) * 3)
  URBANO_15 = Escenario("urbano_15_interurb_5", "-", (0.15,) * 3, (0.05,) * 3)

  print(f"Red: {G.number_of_nodes()} nodos, {G.number_of_edges()} aristas")
  print(f"Muestra: {len(muestra)} trayectos, {muestra.corredor.nunique()} corredores")
  muestra.head()

Red: 5221 nodos, 10243 aristas
Muestra: 16 trayectos, 10 corredores


,corredor,origen,destino,tiempo_seco_csv_s
0,Camino de Ronda,4520478554,534921646,359.0
1,Camino de Ronda,4520478553,534921646,357.0
2,Av. de Andalucía,559691143,534921646,680.0
3,Av. de Andalucía,3435223168,534921646,677.0
4,Ctra. de Málaga,983525310,534921646,646.0


In [4]:
  from trafico_lluvia import criticidad as crit

  K = 1000
  b = crit.betweenness_aristas(G, k=K)

  seco = crit.calcular_criticidad(G, muestra, None, betweenness=b)
  uni = crit.calcular_criticidad(G, muestra, UNIFORME, betweenness=b)

  def top_ids(t, n=20):
      return set(map(tuple, crit.ranking(t, n)[["u", "v", "clave"]].values))

  solape = len(top_ids(seco) & top_ids(uni)) / 20
  base = seco.set_index(["u", "v", "clave"]).criticidad
  otra = uni.set_index(["u", "v", "clave"]).criticidad.reindex(base.index).fillna(0.0)

  res = tl.analizar(G, muestra, [UNIFORME])
  print(f"Enlaces usados: seco={len(seco)}, uniforme={len(uni)}")
  print(f"Solape top-20 seco vs uniforme: {solape:.0%}")
  print(f"Spearman: {base.corr(otra, method='spearman'):.3f}")
  print(f"Trayectos que cambian de ruta: {res.por_trayecto.ruta_cambia.sum()} de {len(muestra)}")
  print(f"Factor de degradación (min/max): {res.por_trayecto.factor_degradacion.min():.4f} / {res.por_trayecto.factor_degradacion.max():.4f}  (1/0.85 = {1/0.85:.4f})")
  crit.ranking(seco, 10)[["via", "corredor", "criticidad"]]



Enlaces usados: seco=403, uniforme=403
Solape top-20 seco vs uniforme: 100%
Spearman: 1.000
Trayectos que cambian de ruta: 0 de 16
Factor de degradación (min/max): 1.1765 / 1.1765  (1/0.85 = 1.1765)


,via,corredor,criticidad
0,Glorieta de la Aviación Española,Av. de Dílar,1.000000
1,Camino de Ronda,Av. de Dílar,0.939836
2,(sin nombre),Av. de Dílar,0.928407
3,(sin nombre),Av. de Dílar,0.920461
4,Camino de Ronda,Av. de Dílar,0.909141
5,Camino de Ronda,Av. de Dílar,0.903297
6,Glorieta de la Aviación Española,Av. de Dílar,0.900677
7,Camino de Ronda,Av. de Dílar,0.899732
8,Camino de Ronda,Av. de Dílar,0.898532
9,Camino de Ronda,Av. de Dílar,0.897051


In [5]:
  nu = crit.calcular_criticidad(G, muestra, URBANO_15, betweenness=b)

  llaves = ["u", "v", "clave"]
  base = seco.set_index(llaves).criticidad
  otra = nu.set_index(llaves).criticidad.reindex(base.index).fillna(0.0)
  res_nu = tl.analizar(G, muestra, [URBANO_15])

  print(f"Enlaces usados: seco={len(seco)}, no uniforme={len(nu)}")
  print(f"Solape top-20: {len(top_ids(seco) & top_ids(nu)) / 20:.0%}")
  print(f"Spearman: {base.corr(otra, method='spearman'):.3f}")
  print(f"Trayectos que cambian de ruta: {res_nu.por_trayecto.ruta_cambia.sum()} de {len(muestra)}")
  print(res_nu.por_trayecto.factor_degradacion.describe().round(4).to_string())

  def top_tabla(t, n=20):
      return crit.ranking(t, n).assign(pos=lambda d: d.index + 1)

  ts, tn = top_tabla(seco), top_tabla(nu)
  clave = lambda d: list(zip(d.u, d.v, d.clave))
  entran = tn[[k not in set(clave(ts)) for k in clave(tn)]]
  salen = ts[[k not in set(clave(tn)) for k in clave(ts)]]
  print("\nENTRAN al top-20 (no estaban en seco):")
  display(entran[["pos", "via", "corredor", "criticidad", "factor_medio"]])
  print("SALEN del top-20 (estaban en seco):")
  display(salen[["pos", "via", "corredor", "criticidad"]])


Enlaces usados: seco=403, no uniforme=403
Solape top-20: 100%
Spearman: 0.999
Trayectos que cambian de ruta: 0 de 16
count    16.0000
mean      1.1542
std       0.0335
min       1.0974
25%       1.1392
50%       1.1765
75%       1.1765
max       1.1765

ENTRAN al top-20 (no estaban en seco):


,pos,via,corredor,criticidad,factor_medio


SALEN del top-20 (estaban en seco):


,pos,via,corredor,criticidad


In [6]:
  filas = []
  for inter in [0.0, 0.05, 0.10, 0.15]:
      esc = Escenario(f"u15_i{int(inter*100)}", "-", (0.15,) * 3, (inter,) * 3)
      t = crit.calcular_criticidad(G, muestra, esc, betweenness=b)
      r = tl.analizar(G, muestra, [esc]).por_trayecto
      o = t.set_index(llaves).criticidad.reindex(base.index).fillna(0.0)
      filas.append({
          "interurbano": f"{inter:.0%}",
          "rutas_cambian": int(r.ruta_cambia.sum()),
          "solape_top10": len(top_ids(seco, 10) & top_ids(t, 10)) / 10,
          "solape_top20": len(top_ids(seco, 20) & top_ids(t, 20)) / 20,
          "solape_top50": len(top_ids(seco, 50) & top_ids(t, 50)) / 50,
          "spearman": round(base.corr(o, method="spearman"), 4),
          "factor_max": round(r.factor_degradacion.max(), 4),
      })
  pd.DataFrame(filas)


,interurbano,rutas_cambian,solape_top10,solape_top20,solape_top50,spearman,factor_max
0,0%,0,1.0,1.0,0.96,0.9987,1.1765
1,5%,0,1.0,1.0,0.98,0.9993,1.1765
2,10%,0,1.0,1.0,0.98,0.9998,1.1765
3,15%,0,1.0,1.0,1.00,1.0000,1.1765


  ## Conclusión

  **Con una caída de velocidad del 15 %, los tramos críticos de Granada son los mismos que en seco.**

  - **Recorte uniforme (−15 % en toda la red):** el ranking de criticidad es idéntico (solape top-20 del
  100 %, Spearman 1,000, 0 de 16 rutas cambian). Es lo esperable: un reescalado uniforme de los tiempos
  no altera cuál es el camino más rápido.
  - **Recorte desigual (−15 % urbano, 0–15 % interurbano):** tampoco aparecen tramos nuevos. Ninguna ruta
  cambia en ningún punto del barrido y el top-10 y el top-20 se mantienen al 100 %. Solo se reordenan
  unos pocos enlaces del borde del top-50 (solape 96–100 %, Spearman ≥ 0,998).
  - **Por qué:** la criticidad que mide el paquete es `betweenness × factor`. La betweenness se calcula
  siempre sobre la red en seco, y la muestra de 16 trayectos periferia→Puerta Real tiene, probablemente,
  poco margen para desviarse. El factor solo pondera por cuánto se degrada cada enlace usado, y eso no
  basta para cambiar el orden de los primeros puestos.

  **Límites:** la conclusión vale para este modelo, no para la ciudad.
  - Solo se modela velocidad, no capacidad ni colas, y la caída del 15 % no es una medición de Granada.
  - La muestra son 16 trayectos hacia un único destino, y solo 403 enlaces entran en la tabla.
  - La betweenness es aproximada (`k=1000`, semilla fija).
  - La explicación de "poco margen para desviarse" no se ha comprobado con otro destino ni con más
  trayectos.
  - Un modelo con congestión, que desvíe tráfico hacia vías alternativas, podría dar un resultado
  distinto. Este análisis no puede descartarlo.
